# 4. Convex functions  [CO 6–9]

TOML-MIRI — Maximum Likelihood Estimation and Convex Optimization, exploration exercises.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, fixed

np.set_printoptions(precision=4, suppress=True)
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.linalg import sqrtm
from scipy.optimize import minimize

def check(label, ok):
    print(f"{'✓' if ok else '✗'}  {label}")

def chord_violations(f, lo, hi, n=4000, seed=0, dim=1):
    # fraction of random (x, y, θ) with f(θx + (1−θ)y) > θf(x) + (1−θ)f(y)   (0 ⇒ no evidence against convexity)
    rng = np.random.default_rng(seed)
    X = rng.uniform(lo, hi, (n, dim)); Y = rng.uniform(lo, hi, (n, dim)); th = rng.uniform(0, 1, (n, 1))
    lhs = f(th * X + (1 - th) * Y); rhs = th[:, 0] * f(X) + (1 - th[:, 0]) * f(Y)
    return np.mean(lhs > rhs + 1e-9 * (1 + np.abs(rhs)))

---
## 4.1 Chords, tangents and epigraphs

**Task.** For $e^{ax}$, $-\log x$, $x\log x$, $x^a$ (on $x>0$) and $|x|$, plot the function, a few chords $\theta f(x) + (1-\theta)f(y)$ and a few tangents $f(x) + f'(x)(y-x)$. Shade the epigraph.

### Prediction

- All five are convex (for $x^a$ with $a\ge1$ or $a\le0$): chords above the graph, tangents below, epigraph a convex region.
- $|x|$ has no tangent at 0; any line through the origin with slope in $[-1,1]$ is a supporting line there.

In [ ]:
def funcs(a=2.0, a_exp=1.0):
    return {
        f"e^({a_exp}x)": (lambda x: np.exp(a_exp * x), lambda x: a_exp * np.exp(a_exp * x), (-2, 2)),
        "−log x": (lambda x: -np.log(x), lambda x: -1 / x, (0.05, 4)),
        "x log x": (lambda x: x * np.log(x), lambda x: np.log(x) + 1, (0.02, 3)),
        f"x^{a}": (lambda x: x**a, lambda x: a * x**(a - 1), (0.05, 3)),
        "|x|": (lambda x: np.abs(x), lambda x: np.sign(x), (-2, 2)),
    }

def plot_function(ax, name, f, df, dom, seed=0):
    rng = np.random.default_rng(seed)
    x = np.linspace(*dom, 400); fx = f(x)
    top = np.nanmax(fx) + 0.3 * (np.nanmax(fx) - np.nanmin(fx) + 1)
    ax.fill_between(x, fx, top, color="#cfe0f3", label="epigraph")
    ax.plot(x, fx, "k", lw=2)
    for _ in range(3):                                          # chords
        p, q = np.sort(rng.uniform(*dom, 2))
        th = np.linspace(0, 1, 50); xs = th * p + (1 - th) * q
        ax.plot(xs, th * f(p) + (1 - th) * f(q), "C3-", lw=1.2)
    for x0 in np.linspace(dom[0] + 0.2 * (dom[1] - dom[0]), dom[1] - 0.2 * (dom[1] - dom[0]), 3):   # tangents
        ax.plot(x, f(x0) + df(x0) * (x - x0), "C2--", lw=1)
    ax.set_ylim(np.nanmin(fx) - 0.2 * (top - np.nanmin(fx)), top); ax.set_xlim(*dom); ax.set_title(name)

fig, axes = plt.subplots(1, 5, figsize=(24, 4.5))
for ax, (name, (f, df, dom)) in zip(axes, funcs().items()):
    plot_function(ax, name, f, df, dom)
axes[0].legend()
plt.show()

### Verify numerically: the three characterisations

In [ ]:
for name, (f, df, dom) in funcs().items():
    rng = np.random.default_rng(1)
    X, Y = rng.uniform(*dom, 5000), rng.uniform(*dom, 5000)
    chord = chord_violations(lambda z: f(z[:, 0]), *dom) == 0
    tangent = np.all(f(Y) >= f(X) + df(X) * (Y - X) - 1e-9)
    xs = np.linspace(dom[0] + 1e-3, dom[1] - 1e-3, 2001); h = 1e-4
    d2 = (f(xs + h) - 2 * f(xs) + f(xs - h)) / h**2
    second = np.all(d2[np.abs(xs) > 1e-2] >= -1e-4)
    check(f"{name:<10} chord above graph: {chord},  tangent below graph: {tangent},  f'' ≥ 0: {second}", chord and tangent and second)

### Explore: $a$ in $x^a$ through $(-1, 0, 0.5, 1, 2)$; add $\sin x$ and $x^3$

In [ ]:
extra = {"sin x": (np.sin, np.cos, (-np.pi, np.pi)), "x³": (lambda x: x**3, lambda x: 3 * x**2, (-1.5, 1.5))}
fig, axes = plt.subplots(1, 7, figsize=(30, 4.2))
for ax, a in zip(axes[:5], [-1, 0, 0.5, 1, 2]):
    f, df, dom = funcs(a=a)[f"x^{a}"]
    plot_function(ax, f"x^{a}", f, df, (0.1, 3))
    v = chord_violations(lambda z: f(z[:, 0]), 0.1, 3); vc = chord_violations(lambda z: -f(z[:, 0]), 0.1, 3)
    ax.set_title(f"x^{a}:  convex {'✓' if v == 0 else '✗'}   concave {'✓' if vc == 0 else '✗'}")
for ax, (name, (f, df, dom)) in zip(axes[5:], extra.items()):
    plot_function(ax, name, f, df, dom)
    v = chord_violations(lambda z: f(z[:, 0]), *dom); vc = chord_violations(lambda z: -f(z[:, 0]), *dom)
    ax.set_title(f"{name}:  convex {'✓' if v == 0 else '✗'}   concave {'✓' if vc == 0 else '✗'}")
plt.show()

---
### Explain

**1. Relate the three characterisations: chord above the graph [CO 2], tangent below the graph [CO 7.1], $f''\ge0$ [CO 7.2].**

They are equivalent descriptions of the same property, each needing more smoothness than the previous one. **Chord above graph** is the definition and needs no derivatives; it says the epigraph is convex (any segment between two points of the epigraph stays in it). **Tangent below graph** ($f(y)\ge f(x)+f'(x)(y-x)$, for differentiable $f$) follows by letting the chord's second endpoint approach the first: the tangent is the limit of chords seen from outside, so it is a *global* under-estimator. It is what makes $\nabla f(x^\star) = 0$ sufficient for a global minimum. **$f''\ge0$** (for twice-differentiable $f$) says the slope $f'$ is non-decreasing, which is exactly what keeps every tangent below the graph. $|x|$ shows the hierarchy: it is convex (chords and epigraph are fine) but has no derivative at 0, where the tangent condition is replaced by *supporting lines* of slope in $[-1, 1]$. $\sin x$ fails all three ($\sin'' = -\sin < 0$ on $(0,\pi)$), $x^3$ fails for $x<0$ (convex only on $[0,\infty)$).

**2. For which $a$ is $x^a$ convex, concave or both?**

On $x>0$: $f'' = a(a-1)x^{a-2}$, whose sign is the sign of $a(a-1)$. **Convex** for $a\le0$ or $a\ge1$ (e.g. $1/x$, $x^2$); **concave** for $0\le a\le1$ (e.g. $\sqrt x$); **both** exactly for $a = 0$ (the constant 1) and $a = 1$ (the line $x$), the affine cases, where $f''\equiv0$. The panels confirm it: $a = -1$ convex, $a = 0$ both, $a = 0.5$ concave, $a = 1$ both, $a = 2$ convex. This matches [CO 8.1].

---
## 4.2 Convexity in $\mathbb R^2$: Hessian test

**Task.** Plot surfaces and contours of: $\|x\|_1$, $\|x\|_2$, $\|x\|_\infty$, $\max(x_1,x_2)$, the log-sum-exp $\log(e^{x_1}+e^{x_2})$, $\|Ax+b\|_2^2$ and $f(x) = x_1x_2$. Where defined, compute the eigenvalues of the Hessian on a grid and colour each point by $\lambda_{\min}(H_f)$.

### Prediction

- All are convex except $x_1x_2$ (a saddle, Hessian $\begin{bmatrix}0&1\\1&0\end{bmatrix}$ with eigenvalues $\pm1$).
- The norms and the max are piecewise linear / conical: Hessian 0 where it exists, undefined on the kinks.
- Log-sum-exp is a smoothed max with a PSD Hessian; $\|Ax+b\|^2$ has constant Hessian $2A^\top A \succeq 0$.

In [ ]:
A2 = np.array([[1.0, 0.5], [0.2, 1.0]]); b2 = np.array([-0.5, 0.3])
f2d = {
    "‖x‖₁": lambda P: np.abs(P[..., 0]) + np.abs(P[..., 1]),
    "‖x‖₂": lambda P: np.hypot(P[..., 0], P[..., 1]),
    "‖x‖∞": lambda P: np.maximum(np.abs(P[..., 0]), np.abs(P[..., 1])),
    "max(x₁, x₂)": lambda P: np.maximum(P[..., 0], P[..., 1]),
    "log(e^x₁ + e^x₂)": lambda P: np.logaddexp(P[..., 0], P[..., 1]),
    "‖Ax + b‖²": lambda P: np.sum((P @ A2.T + b2) ** 2, axis=-1),
    "x₁ x₂": lambda P: P[..., 0] * P[..., 1],
}

def hessian_grid(f, X1, X2, h=1e-3):
    P = np.stack([X1, X2], -1); e1 = np.array([h, 0]); e2 = np.array([0, h])
    fxx = (f(P + e1) - 2 * f(P) + f(P - e1)) / h**2
    fyy = (f(P + e2) - 2 * f(P) + f(P - e2)) / h**2
    fxy = (f(P + e1 + e2) - f(P + e1 - e2) - f(P - e1 + e2) + f(P - e1 - e2)) / (4 * h**2)
    tr, det = fxx + fyy, fxx * fyy - fxy**2
    return tr / 2 - np.sqrt(np.maximum(tr**2 / 4 - det, 0))       # λ_min of the 2×2 Hessian

g = np.linspace(-2, 2, 161)
X1, X2 = np.meshgrid(g, g)
fig = plt.figure(figsize=(28, 8))
lam_min_all = {}
for j, (name, f) in enumerate(f2d.items()):
    F = f(np.stack([X1, X2], -1))
    ax = fig.add_subplot(2, 7, j + 1, projection="3d")
    ax.plot_surface(X1, X2, F, cmap="viridis", lw=0, alpha=0.9); ax.set_title(name)
    ax2 = fig.add_subplot(2, 7, j + 8)
    lam = hessian_grid(f, X1, X2)
    lam = np.where(np.abs(lam) > 50, np.nan, lam)                  # kinks: second differences blow up → not defined
    lam_min_all[name] = lam
    im = ax2.pcolormesh(X1, X2, lam, cmap="RdBu", vmin=-1.5, vmax=1.5, shading="auto")
    ax2.contour(X1, X2, F, levels=10, colors="k", linewidths=0.5); ax2.set_aspect("equal")
    ax2.set_title(f"λ_min(H): min {np.nanmin(lam):.2f}", fontsize=9)
fig.colorbar(im, ax=fig.axes[-1], shrink=0.7)
plt.show()

### Verify numerically

In [ ]:
for name, f in f2d.items():
    v = chord_violations(f, -2, 2, dim=2)
    lam = lam_min_all[name]
    print(f"{name:<18} chord violations: {v:.2%}    min λ_min(H) where defined: {np.nanmin(lam):+.3f}")
check("all functions except x₁x₂ pass the chord test", all(chord_violations(f, -2, 2, dim=2) == 0 for k, f in f2d.items() if k != "x₁ x₂"))
check("x₁x₂ has λ_min(H) = −1 everywhere", np.allclose(lam_min_all["x₁ x₂"], -1, atol=1e-3))
check("Hessian of ‖Ax+b‖² is 2AᵀA (PSD)", np.allclose(np.nanmin(lam_min_all["‖Ax + b‖²"]), np.linalg.eigvalsh(2 * A2.T @ A2).min(), atol=1e-3))
lse = f2d["log(e^x₁ + e^x₂)"]
check("−log-sum-exp is NOT convex (the notes' sign in [CO 8.2])", chord_violations(lambda P: -lse(P), -2, 2, dim=2) > 0)

### $x_1x_2$ along the ray $(0,0) + t(1,-1)$, and the log-sum-exp as a soft max

In [ ]:
t = np.linspace(-2, 2, 201)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.2))
ax1.plot(t, t * (-t), label="f(t, −t) = −t²  (concave along this ray)")
ax1.plot(t, t * 1.0, label="f(t, 1) = t  (linear in x₁ for fixed x₂)")
ax1.legend(); ax1.grid(alpha=0.3); ax1.set_title("x₁x₂ restricted to lines")
x1 = np.linspace(-3, 3, 400)
for beta in [1, 2, 5, 20]:
    ax2.plot(x1, np.logaddexp(beta * x1, 0) / beta, label=f"(1/β) log(e^(βx₁) + e^0), β = {beta}")
ax2.plot(x1, np.maximum(x1, 0), "k--", lw=2, label="max(x₁, 0)")
ax2.legend(fontsize=8); ax2.grid(alpha=0.3); ax2.set_title("log-sum-exp → max as β → ∞  (x₂ = 0)")
plt.show()
P = np.random.default_rng(0).uniform(-3, 3, (10_000, 2))
check("max(x) ≤ log-sum-exp(x) ≤ max(x) + log 2", np.all((lse(P) >= P.max(1)) & (lse(P) <= P.max(1) + np.log(2) + 1e-12)))

---
### Explain

**1. Which functions are convex?**

$\|x\|_1$, $\|x\|_2$, $\|x\|_\infty$, $\max(x_1,x_2)$, $\log(e^{x_1}+e^{x_2})$ and $\|Ax+b\|_2^2$ are convex: every chord test passes and $\lambda_{\min}(H)\ge0$ wherever the Hessian exists. The norms and the max are piecewise linear (or conical, for $\|\cdot\|_2$): their Hessians are 0 (or PSD, $(I - \hat x\hat x^\top)/\|x\|$ for the 2-norm) and they are not differentiable on the kinks (axes, diagonals, the origin), which is why the Hessian map has gaps there; there the convexity is certified by the chord test, or by "pointwise max of linear functions" [CO 9.3]. $\|Ax+b\|^2$ has constant Hessian $2A^\top A\succeq0$. Log-sum-exp has Hessian $\mathrm{diag}(p) - pp^\top \succeq 0$ with $p = \mathrm{softmax}(x)$. **$x_1x_2$ is not convex**: $\lambda_{\min}(H) = -1$ everywhere.

**2. Show along the ray $(0,0)+t(1,-1)$ that $x_1x_2$ is not convex, even though it is convex in each variable separately [CO 9.4].**

$f(t, -t) = -t^2$, a concave parabola, so $f$ restricted to this ray is not convex, and by [CO 6.3] (convex iff convex on every line) $f$ is not convex. Yet for fixed $x_2$, $f = x_2\cdot x_1$ is linear in $x_1$, hence convex, and likewise in $x_2$. Convexity in each coordinate separately only controls the diagonal of the Hessian ($f_{11} = f_{22} = 0 \ge 0$); joint convexity needs the whole Hessian PSD, and the off-diagonal $f_{12} = 1$ makes it indefinite (eigenvalues $\pm1$, a saddle). Directions mixing the two variables are the ones that fail.

**3. Why is the log-sum-exp a "soft" version of max? (Check numerically which sign is convex.)**

$\max_i x_i \le \log\sum_i e^{x_i} \le \max_i x_i + \log n$ (checked numerically for $n = 2$): the largest exponential dominates the sum, and the others add at most $\log n$. Scaling, $\frac1\beta\log\sum e^{\beta x_i}\to\max_i x_i$ as $\beta\to\infty$ (the right panel shows the corner of the max being rounded off less and less). Unlike the max it is smooth everywhere, and its gradient is the softmax vector, a "soft argmax". **Sign:** $+\log\sum e^{x_i}$ is convex (chord test passes, $H\succeq0$); $-\log\sum e^{x_i}$ is *concave* and fails the chord test. So the minus sign written in [CO 8.2] is a typo; the convex function is the log-sum-exp itself.

---
## 4.3 $-\log\det X$ on positive definite matrices

**Task.** On the slice $X = \begin{bmatrix} x & z\\ z & y\end{bmatrix}$ with $z$ fixed (slider), plot $-\log\det X$ as a surface over the region where $X\succ0$. Then pick a random $Z\succ0$ and symmetric $V$, and plot $g(t) = -\log\det(Z+tV)$ on the interval where $Z+tV\succ0$, together with the formula of [CO 8.3]:

$$g(t) = -\log\det Z - \sum_i\log(1+t\lambda_i), \qquad \lambda_i = \mathrm{eig}(Z^{-1/2}VZ^{-1/2}).$$

### Prediction

- A convex bowl over the region $xy > z^2$, rising to $+\infty$ at its boundary (a barrier).
- $g(t)$ is convex and blows up where $1 + t\lambda_i = 0$, i.e. at $t = -1/\lambda_i$.

In [ ]:
def plot_logdet_slice(z=0.5):
    g = np.linspace(0.01, 3, 200)
    Xg, Yg = np.meshgrid(g, g)
    det = Xg * Yg - z**2
    F = np.where(det > 0, -np.log(np.where(det > 0, det, 1)), np.nan)
    fig = go.Figure(go.Surface(x=Xg, y=Yg, z=np.minimum(F, 4), colorscale="Viridis", showscale=False))
    fig.update_layout(title=f"−log det X on the slice z = {z} (clipped at 4)", height=500,
                      scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="−log det"))
    fig.show()

@interact(z=FloatSlider(min=0, max=1.5, step=0.1, value=0.5))
def explore_slice(z):
    plot_logdet_slice(z)

In [ ]:
def random_ray(n, rng, V_kind="indefinite"):
    B = rng.normal(size=(n, n)); Z = B @ B.T + n * np.eye(n) * 0.2
    C = rng.normal(size=(n, n)); V = (C + C.T) / 2
    if V_kind == "PD":
        V = C @ C.T
    return Z, V

def g_numeric(Z, V, ts):
    out = []
    for t in ts:
        sign, logdet = np.linalg.slogdet(Z + t * V)
        out.append(-logdet if sign > 0 and np.linalg.eigvalsh(Z + t * V).min() > 0 else np.nan)
    return np.array(out)

def g_formula(Z, V, ts):
    Zmh = np.linalg.inv(np.real(sqrtm(Z)))
    lam = np.linalg.eigvalsh(Zmh @ V @ Zmh)
    with np.errstate(invalid="ignore", divide="ignore"):
        val = -np.linalg.slogdet(Z)[1] - np.sum(np.log(1 + np.outer(ts, lam)), axis=1)
    return val, lam

def domain(lam):
    lo = max([-1 / l for l in lam if l > 0], default=-np.inf); hi = min([-1 / l for l in lam if l < 0], default=np.inf)
    return lo, hi

rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 3, figsize=(20, 4.5))
for ax, (n, kind) in zip(axes, [(2, "indefinite"), (5, "indefinite"), (5, "PD")]):
    for k in range(4):
        Z, V = random_ray(n, rng, kind)
        _, lam = g_formula(Z, V, np.array([0.0]))
        lo, hi = domain(lam)
        ts = np.linspace(max(lo, -5) + 1e-3, min(hi, 5) - 1e-3, 600)
        gn = g_numeric(Z, V, ts); gf, _ = g_formula(Z, V, ts)
        ax.plot(ts, gn, lw=3, alpha=0.4); ax.plot(ts, gf, "k--", lw=1)
        for b in [lo, hi]:
            if np.isfinite(b) and abs(b) < 5:
                ax.axvline(b, color="r", lw=0.6, ls=":")
    ax.set_title(f"n = {n}, V {kind}: g(t) (color) vs formula (black dashed); red = −1/λᵢ")
    ax.set_xlabel("t"); ax.set_ylim(-15, 15); ax.grid(alpha=0.3)
plt.show()

### Verify numerically

In [ ]:
rng = np.random.default_rng(1)
ok_formula, ok_convex = True, True
for _ in range(200):
    Z, V = random_ray(5, rng)
    _, lam = g_formula(Z, V, np.array([0.0])); lo, hi = domain(lam)
    ts = np.linspace(max(lo, -10) + 1e-2, min(hi, 10) - 1e-2, 300)
    gn = g_numeric(Z, V, ts); gf, _ = g_formula(Z, V, ts)
    ok_formula &= np.allclose(gn, gf, atol=1e-8)
    d2 = np.diff(gn, 2)
    ok_convex &= np.all(d2 >= -1e-9)
check("g(t) = −log det Z − Σ log(1 + tλᵢ) on 200 random rays (n = 5)", ok_formula)
check("g is convex on every ray (second differences ≥ 0; g'' = Σ λᵢ²/(1+tλᵢ)² ≥ 0)", ok_convex)
pairs_ok = True
for _ in range(2000):
    A_, B_ = random_ray(3, rng, "PD"); th = rng.uniform()
    pairs_ok &= -np.linalg.slogdet(th * A_ + (1 - th) * B_)[1] <= th * -np.linalg.slogdet(A_)[1] + (1 - th) * -np.linalg.slogdet(B_)[1] + 1e-10
check("chord inequality for −log det on 2000 random PD pairs (n = 3)", pairs_ok)

---
### Explain

**1. Why is checking convexity along every ray enough [CO 6.3]?**

The definition of convexity only ever involves two points $x, y$ and the segment between them, and that segment lies on the line $x + t(y-x)$. So $f$ is convex iff, for every point $Z$ in the domain and every direction $V$, the one-variable function $g(t) = f(Z+tV)$ is convex on its domain. This turns a matrix problem into scalar calculus: for $-\log\det$ we get $g''(t) = \sum_i\frac{\lambda_i^2}{(1+t\lambda_i)^2}\ge0$ for every $Z\succ0$ and symmetric $V$, which proves convexity on the whole PD cone. The 200 random rays agree with the formula to $10^{-8}$ and are all convex.

**2. Why does $-\log\det X$ act as a barrier for the PD cone?**

$-\log\det X = -\sum_i\log\lambda_i(X)$. As $X$ approaches the boundary of the cone, some eigenvalue $\lambda_i\to0^+$ and $-\log\lambda_i\to+\infty$; along a ray this happens exactly where $1+t\lambda_i\to0$, at $t = -1/\lambda_i$ (red lines). Any descent method that keeps the function finite therefore can never leave $\mathbb S^n_{++}$: the function itself enforces the constraint $X\succ0$. That is how interior-point methods handle the constraint [CO 15.10]. When $V\succ0$, all $\lambda_i>0$, so the ray only hits the boundary going backwards ($t<0$) and stays PD for all $t>0$, where $g\to-\infty$ slowly. With $V$ indefinite the domain is a finite interval with a barrier at both ends.

**3. Relate $\log\det\Sigma$ to the volume of the confidence ellipsoids of a Gaussian.**

The ellipsoid $\{x : (x-\mu)^\top\Sigma^{-1}(x-\mu)\le c^2\}$ is the image of the ball of radius $c$ under $x = \mu + \Sigma^{1/2}u$, so its volume is $\mathrm{vol}(B_c)\cdot\det\Sigma^{1/2} = \mathrm{vol}(B_c)\sqrt{\det\Sigma}$ (product of semi-axes $c\sqrt{\lambda_i}$). Hence $\log\det\Sigma = 2\log\mathrm{vol} + \text{const}$: minimising $\log\det$ of a covariance means making the confidence region as small as possible. This is the D-optimal criterion used for sensor selection [CO 13.5], and the $\log|\Sigma|$ term in the Gaussian log-likelihood penalises "spread-out" models.

---
## 4.4 $\lambda_{\max}(X)$ and $\lambda_{\min}(X)$

**Task.** Use the slice $X = \begin{bmatrix} a+x & y\\ y & a-x\end{bmatrix}$ ($a$ fixed by a slider) and plot the surfaces $\lambda_{\max}(X)$ and $\lambda_{\min}(X)$ over $(x, y)$. Then draw, for 20 unit vectors $u$, the planes $u^\top Xu$ (linear in $X$) and show that $\lambda_{\max}$ is their upper envelope.

Closed form on this slice: $X = aI + \begin{bmatrix} x & y\\ y & -x\end{bmatrix}$ has eigenvalues $a\pm\sqrt{x^2+y^2}$. With $u = (\cos\phi, \sin\phi)$: $u^\top Xu = a + x\cos2\phi + y\sin2\phi$.

### Prediction

- $\lambda_{\max} = a + r$ is an upward cone, $\lambda_{\min} = a - r$ a downward cone, both with apex at $(0, 0, a)$; the planes are tangent to the upper cone along a ray.

In [ ]:
def plot_eig_surfaces(a=1.0, n_planes=20):
    g = np.linspace(-2, 2, 81); XX, YY = np.meshgrid(g, g)
    lmax, lmin = a + np.hypot(XX, YY), a - np.hypot(XX, YY)
    traces = [go.Surface(x=XX, y=YY, z=lmax, colorscale="Reds", showscale=False, opacity=0.9, name="λ_max"),
              go.Surface(x=XX, y=YY, z=lmin, colorscale="Blues", showscale=False, opacity=0.9, name="λ_min")]
    for phi in np.linspace(0, np.pi, n_planes, endpoint=False)[::4]:
        traces.append(go.Surface(x=XX, y=YY, z=a + XX * np.cos(2 * phi) + YY * np.sin(2 * phi),
                                 colorscale=[[0, "gray"], [1, "gray"]], showscale=False, opacity=0.15))
    fig = go.Figure(traces)
    fig.update_layout(height=550, title=f"λ_max (red), λ_min (blue), some planes uᵀXu (gray); a = {a}",
                      scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="eigenvalue"))
    fig.show()

    # 2-D cut y = 0: the lines a + x cos 2φ and their envelopes
    x = np.linspace(-2, 2, 400); phis = np.linspace(0, np.pi, n_planes, endpoint=False)
    lines = a + np.outer(np.cos(2 * phis), x)
    plt.figure(figsize=(8, 4.5))
    plt.plot(x, lines.T, color="gray", lw=0.7)
    plt.plot(x, lines.max(0), "r", lw=3, label="max over 20 planes"); plt.plot(x, a + np.abs(x), "k--", label="λ_max = a + |x|")
    plt.plot(x, lines.min(0), "b", lw=3, label="min over 20 planes"); plt.plot(x, a - np.abs(x), "k:", label="λ_min = a − |x|")
    plt.axhline(0, color="k", lw=0.5); plt.legend(); plt.title("cut y = 0"); plt.grid(alpha=0.3); plt.show()

@interact(a=FloatSlider(min=-1, max=2, step=0.25, value=1.0))
def explore_eigs(a):
    plot_eig_surfaces(a)

### Verify numerically

In [ ]:
rng = np.random.default_rng(0)
a = 1.0
P = rng.uniform(-2, 2, (2000, 2))
eig = np.array([np.linalg.eigvalsh(np.array([[a + p[0], p[1]], [p[1], a - p[0]]])) for p in P])
check("closed form: eigenvalues are a ± √(x² + y²)", np.allclose(eig, np.column_stack([a - np.hypot(*P.T), a + np.hypot(*P.T)])))
phis = np.linspace(0, np.pi, 2000, endpoint=False)
env = (a + np.outer(P[:, 0], np.cos(2 * phis)) + np.outer(P[:, 1], np.sin(2 * phis))).max(axis=1)
check("λ_max = sup over unit u of uᵀXu (dense envelope)", np.allclose(env, eig[:, 1], atol=1e-5))

viol_max, viol_min = 0, 0
for _ in range(1000):
    Xa, Ya = rng.normal(size=(5, 5)), rng.normal(size=(5, 5)); Xs, Ys = (Xa + Xa.T) / 2, (Ya + Ya.T) / 2
    th = rng.uniform()
    M = th * Xs + (1 - th) * Ys
    viol_max += np.linalg.eigvalsh(M)[-1] > th * np.linalg.eigvalsh(Xs)[-1] + (1 - th) * np.linalg.eigvalsh(Ys)[-1] + 1e-12
    viol_min += np.linalg.eigvalsh(M)[0] < th * np.linalg.eigvalsh(Xs)[0] + (1 - th) * np.linalg.eigvalsh(Ys)[0] - 1e-12
check("λ_max(θX + (1−θ)Y) ≤ θλ_max(X) + (1−θ)λ_max(Y) on 1000 random 5×5 pairs", viol_max == 0)
check("λ_min(θX + (1−θ)Y) ≥ θλ_min(X) + (1−θ)λ_min(Y) (concave)", viol_min == 0)

---
### Explain

**1. Why is a pointwise supremum of linear functions convex [CO 9.3]?**

Each $X\mapsto u^\top Xu$ is linear, so its epigraph is a halfspace. The epigraph of $\sup_u u^\top Xu$ is the set of $(X, t)$ with $t\ge u^\top Xu$ for **all** $u$, the intersection of all those halfspaces, and an intersection of convex sets is convex. Directly: $\lambda_{\max}(\theta X+(1-\theta)Y) = \sup_u[\theta u^\top Xu + (1-\theta)u^\top Yu] \le \theta\sup_u u^\top Xu + (1-\theta)\sup_u u^\top Yu$, because optimising the two terms separately can only do better. The 1000 random $5\times5$ pairs confirm the inequality, and the cut $y = 0$ shows the envelope of 20 lines approaching the V-shape $a + |x|$.

**2. Why is $\lambda_{\min}$ concave?**

$\lambda_{\min}(X) = \inf_{\|u\|=1}u^\top Xu$ is a pointwise *infimum* of linear (hence concave) functions, which is concave by the mirror argument (its hypograph is an intersection of halfspaces). Equivalently $\lambda_{\min}(X) = -\lambda_{\max}(-X)$, minus a convex function. On the slice it is the downward cone $a - \sqrt{x^2+y^2}$.

**3. Where are these functions not differentiable, and what happens to the eigenvectors there?**

At the apex $x = y = 0$, i.e. $X = aI$, where the two eigenvalues **coincide**. Everywhere else the surfaces are smooth cones. At the apex the cone has a kink: approaching from different directions gives different slopes, so there is no gradient. The eigenvectors also stop being defined: for $X = aI$ *every* unit vector is an eigenvector. As $(x,y)\to0$ along the direction angle $\psi$, the top eigenvector is $(\cos\frac\psi2, \sin\frac\psi2)$, which depends on the direction of approach, so it has no limit. In general, $\lambda_{\max}$ is differentiable exactly where the largest eigenvalue is simple, with gradient $v_1v_1^\top$; at repeated eigenvalues only *subgradients* exist (any $uu^\top$ with $u$ in the eigenspace).

**4. Restrict to $X\succ0$: which part of the slice is that, and how does $\lambda_{\min}(X) > 0$ describe the PD cone?**

$X\succ0 \iff \lambda_{\min}(X) > 0 \iff a - \sqrt{x^2+y^2} > 0$: the **open disc of radius $a$** around the origin (empty if $a \le 0$). It is the region where the blue cone is above zero. In general $\mathbb S^n_{++} = \{X : \lambda_{\min}(X) > 0\}$ is a strict superlevel set of the concave function $\lambda_{\min}$, which gives another proof that the PD cone is convex.

---
## 4.5 $\mathrm{tr}(SX)$ and the Gaussian log-likelihood

**Task.** Fix $S\succ0$ ($2\times2$). On the PSD cone of Exercise 3.3, plot the level sets of the linear function $X\mapsto\mathrm{tr}(SX)$. Then plot, on a 2-D slice, the function

$$h(X) = -\log\det X + \mathrm{tr}(SX), \qquad X\succ0,$$

and find its minimiser numerically. Check that it is $X^\star = S^{-1}$.

### Prediction

- $\mathrm{tr}(SX) = s_{11}x + s_{22}y + 2s_{12}z$ is linear: its level sets are planes, and on the PSD cone they cut out ellipses (bounded slices), because $S\succ0$.
- $h$ is convex with a unique minimiser $S^{-1}$ (gradient $-X^{-1} + S = 0$).

In [ ]:
S = np.array([[2.0, 0.6], [0.6, 1.0]])

r = np.linspace(0, 1.5, 30); ph = np.linspace(0, 2 * np.pi, 60); R, PH = np.meshgrid(r, ph)
Xb, Yb, Zb = R + R * np.cos(PH), R - R * np.cos(PH), R * np.sin(PH)
traces = [go.Surface(x=Xb, y=Yb, z=Zb, opacity=0.25, colorscale="Blues", showscale=False)]
for c in [0.5, 1.0, 2.0]:
    # points of the cone boundary with tr(SX) = c: scale each boundary ray so that tr(S·X) = c
    ray = np.stack([1 + np.cos(ph), 1 - np.cos(ph), np.sin(ph)])
    tr_ray = S[0, 0] * ray[0] + S[1, 1] * ray[1] + 2 * S[0, 1] * ray[2]
    curve = ray * (c / tr_ray)
    traces.append(go.Scatter3d(x=curve[0], y=curve[1], z=curve[2], mode="lines", line=dict(width=6), name=f"tr(SX) = {c}"))
fig = go.Figure(traces)
fig.update_layout(height=500, title="level sets of tr(SX) on the boundary of the PSD cone",
                  scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z", aspectmode="cube"))
fig.show()

S_inv = np.linalg.inv(S)
def h(x, y, z):
    det = x * y - z**2
    return np.where((det > 0) & (x > 0), -np.log(np.where(det > 0, det, 1)) + S[0, 0] * x + S[1, 1] * y + 2 * S[0, 1] * z, np.nan)
g = np.linspace(0.05, 2.5, 300); GX, GY = np.meshgrid(g, g)
H = h(GX, GY, S_inv[0, 1])
plt.figure(figsize=(7, 6))
cs = plt.contour(GX, GY, H, levels=np.nanmin(H) + np.array([0.02, 0.1, 0.3, 0.6, 1, 2, 3]), cmap="viridis"); plt.clabel(cs, fmt="%.2f")
plt.plot(S_inv[0, 0], S_inv[1, 1], "r*", ms=15, label="S⁻¹ (x, y entries)")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.title(f"h on the slice z = (S⁻¹)₁₂ = {S_inv[0, 1]:.3f}")
plt.show()

def h_chol(v):     # parametrise X = LLᵀ with L lower triangular, diag > 0 via exp: unconstrained and always PD
    L = np.array([[np.exp(v[0]), 0], [v[1], np.exp(v[2])]]); X = L @ L.T
    return -np.linalg.slogdet(X)[1] + np.trace(S @ X)
res = minimize(h_chol, x0=np.zeros(3))
L = np.array([[np.exp(res.x[0]), 0], [res.x[1], np.exp(res.x[2])]]); X_star = L @ L.T
print("numerical minimiser X* =\n", X_star, "\nS⁻¹ =\n", S_inv)

### Verify numerically

In [ ]:
check("argmin h = S⁻¹", np.allclose(X_star, S_inv, atol=1e-5))
check("∇h(S⁻¹) = −(S⁻¹)⁻¹ + S = 0", np.allclose(-np.linalg.inv(S_inv) + S, 0))
check("min h = n + log det S  (= 2 + log det S)", np.isclose(res.fun, 2 + np.log(np.linalg.det(S))))
rng = np.random.default_rng(0)
def rand_psd(n):
    B = rng.normal(size=(n, n)); return B @ B.T
tr_vals = [np.trace(rand_psd(3) @ rand_psd(3)) for _ in range(10_000)]
check("tr(SX) ≥ 0 for 10,000 random PSD pairs (n = 3)", min(tr_vals) >= 0)
S_ind, X_psd = np.diag([1.0, -1.0]), np.diag([0.0, 1.0])
check(f"indefinite S = diag(1, −1), PSD X = diag(0, 1): tr(SX) = {np.trace(S_ind @ X_psd)} < 0", np.trace(S_ind @ X_psd) < 0)
# the ML problem in Σ is NOT concave: scalar case ℓ(s) = −log s − S/s
s = np.linspace(0.2, 6, 500); ell = -np.log(s) - 1.0 / s
check("ℓ(σ²) = −log σ² − S/σ² is not concave (second difference > 0 for σ² > 2S)", np.any(np.diff(ell, 2) > 0))

---
### Explain

**1. Why is $\mathrm{tr}(SX)$ both convex and concave?**

It is **linear** in $X$: $\mathrm{tr}(S(\alpha X + \beta Y)) = \alpha\,\mathrm{tr}(SX) + \beta\,\mathrm{tr}(SY)$; on the $2\times2$ slice it is $s_{11}x + s_{22}y + 2s_{12}z$. A linear (affine) function satisfies the chord inequality with equality, so it is convex and $-$it is convex too, i.e. it is also concave. Its level sets are hyperplanes (the planes cutting the cone in the figure).

**2. Why is $\mathrm{tr}(SX)\ge0$ for PSD $S, X$ [CO 12.4]?**

Write $S = \sum_i\sigma_iu_iu_i^\top$ with $\sigma_i\ge0$. Then $\mathrm{tr}(SX) = \sum_i\sigma_i\mathrm{tr}(u_iu_i^\top X) = \sum_i\sigma_i\,u_i^\top Xu_i\ge0$, because each $u_i^\top Xu_i\ge0$ ($X\succeq0$). Equivalently $\mathrm{tr}(SX) = \|S^{1/2}X^{1/2}\|_F^2$. So the PSD cone is *self-dual*: its "dual cone" under the inner product $\langle S, X\rangle = \mathrm{tr}(SX)$ is itself. This is what gives weak duality for SDPs. It fails as soon as one matrix is indefinite: $S = \mathrm{diag}(1,-1)$, $X = \mathrm{diag}(0,1)$ gives $-1$.

**3. Using Exercise 2.4, explain why minimising $h$ with $S$ the sample covariance is exactly the ML estimation of the precision matrix [CO 13.4]: is the ML problem concave in $\Sigma$, in $\Lambda = \Sigma^{-1}$, or in both?**

From 2.4, $\ell(\mu_{ML},\Lambda) = -\frac{mn}{2}\log2\pi + \frac m2\big[\log\det\Lambda - \mathrm{tr}(\Lambda S)\big] = \text{const} - \frac m2 h(\Lambda)$. Maximising the likelihood over $\Lambda\succ0$ is therefore exactly minimising $h(\Lambda) = -\log\det\Lambda + \mathrm{tr}(S\Lambda)$, whose minimiser $\Lambda^\star = S^{-1}$ (checked numerically) gives $\Sigma_{ML} = S$, consistent with 2.4. **Concavity:** in $\Lambda$ the log-likelihood is concave ($\log\det$ is concave and $-\mathrm{tr}(S\Lambda)$ is linear), so this is a convex problem, and adding a convex penalty $\lambda\|\Lambda\|_1$ keeps it convex: the graphical lasso [CO 13.4]. In $\Sigma$ it is $\text{const} - \frac m2[\log\det\Sigma + \mathrm{tr}(S\Sigma^{-1})]$, which is **not** concave: already in 1-D, $-\log s - S/s$ has positive second derivative for $s > 2S$ (checked), the same "curling up" seen in Exercise 2.1. So the ML problem is concave in $\Lambda$ only; the maximiser is the same point in both parametrisations, but only the precision parametrisation makes it a convex optimisation problem.

---
## 4.6 Operations that preserve convexity and Jensen

**Task.** Build functions with the rules of [CO 9]: $\|Ax+b\|_2^2$, $\max_i(a_i^\top x + b_i)$ (plot the piecewise-linear surface), $e^{g(x)}$ with $g$ convex, $1/g(x)$ with $g$ concave and positive. Then, for $f(x) = e^x$ and a random discrete $X$, check $f(EX)\le Ef(X)$ [CO 9.7] and visualise it with the chord picture.

### Prediction

- All four are convex. Jensen: the point $(EX, Ef(X))$ is a convex combination of points on the graph, so it lies inside the convex hull, above the curve.

In [ ]:
rng = np.random.default_rng(0)
A6, b6 = rng.normal(size=(3, 2)), rng.normal(size=3)
a_i, b_i = rng.normal(size=(5, 2)), rng.normal(size=5)
built = {
    "‖Ax + b‖²  (affine map of a convex fn)": lambda P: np.sum((P @ A6.T + b6) ** 2, axis=-1),
    "maxᵢ(aᵢᵀx + bᵢ)  (pointwise max)": lambda P: np.max(P @ a_i.T + b_i, axis=-1),
    "exp(‖x‖² − 1)  (convex increasing ∘ convex)": lambda P: np.exp(np.sum(P**2, axis=-1) - 1),
    "1/(6 − ‖x‖²)  (1/g, g concave > 0)": lambda P: 1 / (6 - np.sum(P**2, axis=-1)),
}
g = np.linspace(-1.5, 1.5, 121); X1, X2 = np.meshgrid(g, g); P = np.stack([X1, X2], -1)
fig = plt.figure(figsize=(24, 5))
for j, (name, f) in enumerate(built.items()):
    ax = fig.add_subplot(1, 4, j + 1, projection="3d"); ax.plot_surface(X1, X2, f(P), cmap="viridis", lw=0)
    ax.set_title(name, fontsize=9)
plt.show()
for name, f in built.items():
    check(f"{name}: chord test passes", chord_violations(f, -1.5, 1.5, dim=2) == 0)

In [ ]:
xs = rng.uniform(-2, 2, 5); ps = rng.dirichlet(np.ones(5))
EX, EfX = ps @ xs, ps @ np.exp(xs)
t = np.linspace(-2.2, 2.2, 300)
plt.figure(figsize=(8, 5))
plt.plot(t, np.exp(t), "k", lw=2, label="f(x) = eˣ")
plt.scatter(xs, np.exp(xs), s=400 * ps, c="C0", zorder=3, label="(xᵢ, f(xᵢ)), size ∝ pᵢ")
order = np.argsort(xs); hull_pts = np.column_stack([xs[order], np.exp(xs[order])])
plt.fill(np.r_[hull_pts[:, 0], hull_pts[::-1, 0]], np.r_[hull_pts[:, 1], np.interp(hull_pts[::-1, 0], xs[order], np.exp(xs[order]))],
         alpha=0.15)
plt.plot(EX, EfX, "r*", ms=16, label=f"(E X, E f(X)) = ({EX:.2f}, {EfX:.2f})")
plt.plot(EX, np.exp(EX), "ko", ms=8, label=f"(E X, f(E X)) = ({EX:.2f}, {np.exp(EX):.2f})")
plt.vlines(EX, np.exp(EX), EfX, colors="r", linestyles=":")
plt.legend(); plt.grid(alpha=0.3); plt.title("Jensen: f(E X) ≤ E f(X) for convex f"); plt.show()
check("f(E X) ≤ E f(X) for this X", np.exp(EX) <= EfX)
jensen_ok = True
for _ in range(10_000):
    v = rng.uniform(-3, 3, 6); w = rng.dirichlet(np.ones(6))
    jensen_ok &= np.exp(w @ v) <= w @ np.exp(v) + 1e-12
check("Jensen holds for 10,000 random discrete X", jensen_ok)

# a convex h composed with a convex g that is NOT convex: h(u) = e^{−u} (convex, decreasing), g(x) = x²
x = np.linspace(-3, 3, 400)
plt.figure(figsize=(7, 3.5)); plt.plot(x, np.exp(-x**2)); plt.title("h(g(x)) = exp(−x²): convex h (decreasing) ∘ convex g  →  not convex")
plt.grid(alpha=0.3); plt.show()
check("exp(−x²) fails the chord test", chord_violations(lambda z: np.exp(-z[:, 0] ** 2), -3, 3) > 0)

---
### Explain

**1. For each function, which rule certifies convexity?**

- $\|Ax+b\|_2^2$: $\|\cdot\|_2^2$ is convex and composing with an **affine map** preserves convexity [CO 9.1].
- $\max_i(a_i^\top x + b_i)$: each piece is affine (convex); the **pointwise maximum** of convex functions is convex [CO 9.3]. The surface is a "roof" of planar facets.
- $e^{g(x)}$ with $g(x) = \|x\|^2 - 1$: **scalar composition** [CO 9.5] with $h = \exp$ convex and non-decreasing and $g$ convex; indeed $f'' = h''(g')^2 + h'g'' \ge 0$ (both terms $\ge 0$).
- $1/g(x)$ with $g = 6-\|x\|^2$ concave and positive on the plotted square $[-1.5, 1.5]^2$ (where $\|x\|^2\le4.5$): composition with $h(u) = 1/u$, which on $u>0$ is convex and **non-increasing**, with $g$ concave: $h''(g')^2\ge0$ and $h'g''\ge0$ (negative times negative).

All four pass the chord test.

**2. Find an example where composing a convex $h$ with a convex $g$ is *not* convex (hint: $h$ decreasing).**

$h(u) = e^{-u}$ (convex, decreasing) and $g(x) = x^2$ (convex): $h(g(x)) = e^{-x^2}$, the Gaussian bump, which is concave near 0 and fails the chord test. In the formula $f'' = h''(g')^2 + h'g''$, the first term is $\ge0$ but the second is $h'g'' < 0$ (decreasing $h$ times convex $g$), and near $x = 0$ (where $g' = 0$) it wins: $f''(0) = -2 < 0$. The rule needs $h$ non-decreasing when $g$ is convex.

**3. Check $f(EX)\le Ef(X)$ [CO 9.7] and explain the chord picture.**

For the plotted $X$ (5 values, random probabilities) $f(EX) < Ef(X)$, and the inequality held for 10,000 random discrete distributions. Picture: the points $(x_i, f(x_i))$ lie on the convex curve; $(EX, Ef(X)) = \sum_ip_i(x_i, f(x_i))$ is a convex combination of them, so it lies in their convex hull, which, because the curve is convex, sits **above** the curve. The vertical red gap between the star and the black dot at $x = EX$ is $Ef(X) - f(EX)\ge0$. With two points this is literally the chord inequality; Jensen is its generalisation to any number of points (and to densities).